# Single-run outputs

Summer4 port of `tweak_cases_calib` `notebooks/03-InspectRun.ipynb`. Same
figure groups: population structure, infection source, and mixing.

Uses `run(params)` — corrected model, folded spline contact, smooth entry —
not the golden-matching path. Mixing heatmaps are recomputed with
`spline_mixing` (summer4 does not save `computed_values["dynamic_mm"]`).

`MODE`:
- `base` — `BASE_PARAMS`
- `optimise` — short Adam fit of the calibration likelihood (see also
  `notebooks/06-Optimisation.ipynb`)
- `random` / `map` — draw from a saved ArviZ `.nc` under `outputs/` or
  `outputs/local/`

Use the **vacf (pixi)** kernel. If this folder is the notebook
cwd, the `plots` helper imports via the cell below.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import optax
from jax import jit, nn, value_and_grad
from jax import numpy as jnp

from vacf.calibration import (
    TERM_NAMES,
    build_mixing_cases,
    compile_case,
    make_log_likelihood,
)
from vacf.constants import END_TIME, ISO3, LOCAL_OUTPUT_PATH, OUTPUT_PATH
from vacf.model import run
from vacf.parameters import BASE_PARAMS, PARAM_BOUNDS, get_nuts_init_values

# Allow `from plots import ...` when the kernel cwd is the repo root or this folder.
_here = Path.cwd()
for candidate in (_here, _here / "notebooks" / "summer3", _here / "summer3"):
    if (candidate / "plots.py").is_file():
        sys.path.insert(0, str(candidate))
        break

from plots import (
    load_group_popsize,
    plot_age_population_comparison,
    plot_comp_distributions,
    plot_dynamic_mixing_matrix,
    plot_infection_source_props,
    plot_mean_contacts_by_age,
    plot_mixing_target_comparison,
    plot_single_age_s_matrix_components,
)


In [ ]:
MODE = "base"  # "base", "optimise", "random", or "map"

CALIB_SOURCE = "local"  # "local" or "remote"
RUN_ID = None  # e.g. "20260917T1241Z" stamp, or None for newest .nc
FOLDER = None  # remote task folder under OUTPUT_PATH, or None for newest
SAMPLE_SEED = 0


In [ ]:
import arviz as az

calib_names = [p for p in PARAM_BOUNDS if p != "mixing_dist_sd"]


def _newest_nc(directory: Path) -> Path:
    files = sorted(directory.glob("*.nc"), key=lambda p: p.stat().st_mtime)
    if not files:
        raise FileNotFoundError(f"No .nc under {directory}")
    return files[-1]


def load_idata():
    if CALIB_SOURCE == "local":
        directory = LOCAL_OUTPUT_PATH
        path = directory / f"{RUN_ID}.nc" if RUN_ID else _newest_nc(directory)
    elif CALIB_SOURCE == "remote":
        root = OUTPUT_PATH
        if FOLDER is not None:
            directory = root / FOLDER
        else:
            dirs = sorted(
                (p for p in root.iterdir() if p.is_dir() and any(p.glob("*.nc"))),
                key=lambda p: p.stat().st_mtime,
            )
            if not dirs:
                raise FileNotFoundError(f"No run dirs under {root}")
            directory = dirs[-1]
        path = directory / f"{RUN_ID}.nc" if RUN_ID else _newest_nc(directory)
    else:
        raise ValueError(f"Unknown CALIB_SOURCE: {CALIB_SOURCE}")
    if not path.exists():
        raise FileNotFoundError(path)
    return az.from_netcdf(path), path


group_popsize = load_group_popsize(ISO3)


In [ ]:
run_params = dict(BASE_PARAMS)
opt_history = None
sample_info = None

if MODE == "base":
    pass

elif MODE == "optimise":
    case = next(item for item in build_mixing_cases() if item.name == "spline")
    compiled = compile_case(case)
    evaluate = make_log_likelihood(compiled, mixing=case.mixing, fold=case.fold)
    names = tuple(PARAM_BOUNDS)
    low = jnp.asarray([PARAM_BOUNDS[n][0] for n in names])
    high = jnp.asarray([PARAM_BOUNDS[n][1] for n in names])
    start = get_nuts_init_values()

    def constrain(raw):
        return low + (high - low) * nn.sigmoid(raw)

    def unconstrain(value):
        unit = (value - low) / (high - low)
        return jnp.log(unit) - jnp.log1p(-unit)

    def unpack(raw):
        params = dict(BASE_PARAMS)
        for name, value in zip(names, constrain(raw), strict=True):
            params[name] = value
        return params

    raw = unconstrain(jnp.asarray([start[n] for n in names]))

    def loss(point):
        return -evaluate(unpack(point))["total"]

    optimizer = optax.adam(0.01)
    opt_state = optimizer.init(raw)

    @jit
    def step(point, state):
        value, grad = value_and_grad(loss)(point)
        updates, state = optimizer.update(grad, state, point)
        return optax.apply_updates(point, updates), state, value

    opt_history = []
    for i in range(40):
        raw, opt_state, value = step(raw, opt_state)
        opt_history.append(float(value))
        if i % 10 == 0:
            print(f"step {i:3d}  nll {opt_history[-1]:.4f}")
    run_params = unpack(raw)
    run_params = {k: float(v) if hasattr(v, "item") else v for k, v in run_params.items()}
    print("optimise done", {n: run_params[n] for n in names})

elif MODE in {"random", "map"}:
    idata, nc_path = load_idata()
    stacked = idata.posterior.stack(sample=("chain", "draw"))
    if hasattr(stacked, "dataset"):
        stacked = stacked.dataset
    n_samples = int(stacked.sizes["sample"])
    if MODE == "random":
        rng = np.random.default_rng(SAMPLE_SEED)
        idx = int(rng.integers(n_samples))
    else:
        lp = idata.sample_stats["lp"].stack(sample=("chain", "draw"))
        idx = int(np.asarray(lp).argmax())
    sample = stacked.isel(sample=idx)
    drawn = {k: float(sample[k].values) for k in sample.data_vars}
    run_params = dict(BASE_PARAMS)
    run_params.update(drawn)
    sample_info = {"path": str(nc_path), "index": idx}
    print(sample_info)
    print({k: run_params[k] for k in PARAM_BOUNDS})

else:
    raise ValueError(MODE)

result = run(run_params)
print("ran", MODE)


# Population structure


In [ ]:
plot_comp_distributions(result, group_popsize, 1950.0, END_TIME)


In [ ]:
plot_age_population_comparison(result, group_popsize, range(1970, 2021, 10))


# Infection source


In [ ]:
plot_infection_source_props(result, 1950.0, END_TIME)


# Mixing


In [ ]:
plot_dynamic_mixing_matrix(run_params, 1970.0, 10.0, 3)


In [ ]:
plot_mixing_target_comparison(run_params, ISO3)


In [ ]:
plot_single_age_s_matrix_components(run_params)


In [ ]:
plot_mean_contacts_by_age(run_params, ISO3)
